# Merge & Normalize — All 4 Labeled Cases
Merges `single_case1`, `single_case2`, `complex_case1`, `complex_case2` into a single file with a `case` column,  
then re-normalizes all feature columns using `StandardScaler` fitted on normal rows from `single_case1` only.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

BASE        = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3\data'
LABELED_DIR = os.path.join(BASE, 'labeled')
OUT_DIR     = os.path.join(BASE, 'processed')
os.makedirs(OUT_DIR, exist_ok=True)

CASES = ['single_case1', 'single_case2', 'complex_case1', 'complex_case2']

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate',
    'container_cpu_system_seconds_rate',
    'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss',
    'container_memory_cache',
]

print('Setup complete.')
print('Cases    :', CASES)
print('Features :', FEATURE_COLS)

Setup complete.
Cases    : ['single_case1', 'single_case2', 'complex_case1', 'complex_case2']
Features : ['container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache']


## Step 1 — Load All 4 Labeled Files

In [2]:
dfs = []

for case in CASES:
    path = os.path.join(LABELED_DIR, f'{case}_labeled.csv')
    df   = pd.read_csv(path, low_memory=False)
    df.insert(2, 'case', case)   # add case column right after cmdb_id
    dfs.append(df)
    print(f"{case:20s} : {len(df):>7,} rows  |  anomalies: {df['label'].sum():>5,}  ({df['label'].mean()*100:.2f}%)")

print(f"\nColumns: {list(dfs[0].columns)}")

single_case1         :  74,196 rows  |  anomalies:   116  (0.16%)
single_case2         :  45,711 rows  |  anomalies:    56  (0.12%)
complex_case1        : 318,418 rows  |  anomalies: 1,242  (0.39%)
complex_case2        :  94,905 rows  |  anomalies:   812  (0.86%)

Columns: ['timestamp', 'cmdb_id', 'case', 'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache', 'label', 'failure_type']


## Step 2 — Concatenate Into One DataFrame

In [3]:
full = pd.concat(dfs, ignore_index=True)

print(f"Total rows      : {len(full):,}")
print(f"Total anomalies : {full['label'].sum():,}  ({full['label'].mean()*100:.2f}%)")
print(f"Cases           : {full['case'].unique().tolist()}")
print(f"Containers      : {full['cmdb_id'].nunique()} unique")

full.head(3)

Total rows      : 533,230
Total anomalies : 2,226  (0.42%)
Cases           : ['single_case1', 'single_case2', 'complex_case1', 'complex_case2']
Containers      : 27 unique


,timestamp,cmdb_id,case,container_cpu_usage_seconds_rate,container_cpu_system_seconds_rate,container_cpu_user_seconds_rate,container_memory_usage_bytes,container_memory_working_set_bytes,container_memory_rss,container_memory_cache,label,failure_type
0,1719252015,observe.cartservice-0,single_case1,0.658709,0.657895,0.549296,0.065104,0.065104,0.050802,0.0,0,NaN
1,1719252030,observe.cartservice-0,single_case1,0.000000,0.000000,0.000000,0.065104,0.065104,0.050802,0.0,0,NaN
2,1719252045,observe.cartservice-0,single_case1,0.000000,0.000000,0.000000,0.065104,0.065104,0.050802,0.0,0,NaN


## Step 3 — Re-Normalize with StandardScaler

**Why re-normalize?**  
Each labeled file was normalized independently (per-case min-max).  
A value of `0.8` in `single_case1` represents a different absolute level than `0.8` in `complex_case1`.  
We re-normalize using `StandardScaler` (mean=0, std=1) fitted **only on normal rows from `single_case1`** — the training split — so no test data leaks into the scaler.

**Why StandardScaler instead of Min-Max?**  
Min-Max would clip anomalies to exactly 1.0, hiding the anomaly signal.  
Z-score lets anomalies exceed the normal range, giving stronger reconstruction error contrast in the VAE.

In [4]:
# Fit ONLY on normal rows from single_case1 (no data leakage from test cases)
train_normal = full[(full['case'] == 'single_case1') & (full['label'] == 0)]
print(f"Fitting scaler on {len(train_normal):,} normal rows from single_case1 ...")

scaler = StandardScaler()
scaler.fit(train_normal[FEATURE_COLS])

# Show learned parameters
scaler_params = pd.DataFrame({
    'feature' : FEATURE_COLS,
    'mean'    : scaler.mean_.round(4),
    'std'     : scaler.scale_.round(4),
})
print("\nScaler parameters (fitted on single_case1 normal rows):")
display(scaler_params)

Fitting scaler on 74,080 normal rows from single_case1 ...

Scaler parameters (fitted on single_case1 normal rows):


,feature,mean,std
0,container_cpu_usage_seconds_rate,0.1145,0.2426
1,container_cpu_system_seconds_rate,0.0995,0.2151
2,container_cpu_user_seconds_rate,0.0931,0.2076
3,container_memory_usage_bytes,0.3819,0.2529
4,container_memory_working_set_bytes,0.3801,0.2517
5,container_memory_rss,0.3593,0.2369
6,container_memory_cache,0.2105,0.3439


In [5]:
# Transform all rows (train + val + test)
full[FEATURE_COLS] = scaler.transform(full[FEATURE_COLS])

print("Feature stats AFTER normalization:")
display(full[FEATURE_COLS].describe().round(3))

Feature stats AFTER normalization:


,container_cpu_usage_seconds_rate,container_cpu_system_seconds_rate,container_cpu_user_seconds_rate,container_memory_usage_bytes,container_memory_working_set_bytes,container_memory_rss,container_memory_cache
count,533230.000,533230.000,533230.000,533230.000,533230.000,533230.000,533230.000
mean,-0.149,-0.092,-0.137,0.613,0.642,0.787,0.553
std,0.787,0.867,0.806,1.255,1.258,1.330,1.054
min,-0.472,-0.463,-0.449,-1.510,-1.510,-1.516,-0.612
25%,-0.472,-0.463,-0.449,-0.562,-0.537,-0.319,-0.597
50%,-0.472,-0.463,-0.449,0.662,0.687,0.809,0.394
75%,-0.472,-0.463,-0.449,1.812,1.863,2.103,1.430
max,3.650,4.187,4.369,2.444,2.462,2.704,2.296


## Step 4 — Save to `data/processed/all_cases_labeled.csv`

In [6]:
out_path = os.path.join(OUT_DIR, 'all_cases_labeled.csv')
full.to_csv(out_path, index=False)

size_mb = os.path.getsize(out_path) / (1024 * 1024)
print(f"Saved  : {out_path}")
print(f"Shape  : {full.shape[0]:,} rows x {full.shape[1]} columns")
print(f"Size   : {size_mb:.1f} MB")

Saved  : c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\all_cases_labeled.csv
Shape  : 533,230 rows x 12 columns
Size   : 98.0 MB


## Step 5 — Verify Output

In [7]:
verify = pd.read_csv(out_path, low_memory=False)

print("Columns:", list(verify.columns))
print()

print("Rows per case:")
summary = verify.groupby('case').agg(
    rows      = ('label', 'count'),
    anomalies = ('label', 'sum'),
).copy()
summary['anomaly_pct'] = (summary['anomalies'] / summary['rows'] * 100).round(2).astype(str) + '%'
display(summary)

Columns: ['timestamp', 'cmdb_id', 'case', 'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate', 'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache', 'label', 'failure_type']

Rows per case:


,rows,anomalies,anomaly_pct
case,,,
complex_case1,318418,1242,0.39%
complex_case2,94905,812,0.86%
single_case1,74196,116,0.16%
single_case2,45711,56,0.12%


In [8]:
print("Failure type distribution:")
display(verify['failure_type'].value_counts(dropna=False).rename('count').to_frame())

Failure type distribution:


,count
failure_type,
NaN,531004
loss,627
memory,483
delay,458
pod-failure,414
cpu,244


In [9]:
print("Feature value ranges after normalization:")
display(verify[FEATURE_COLS].agg(['min', 'mean', 'max']).round(3))

Feature value ranges after normalization:


,container_cpu_usage_seconds_rate,container_cpu_system_seconds_rate,container_cpu_user_seconds_rate,container_memory_usage_bytes,container_memory_working_set_bytes,container_memory_rss,container_memory_cache
min,-0.472,-0.463,-0.449,-1.510,-1.510,-1.516,-0.612
mean,-0.149,-0.092,-0.137,0.613,0.642,0.787,0.553
max,3.650,4.187,4.369,2.444,2.462,2.704,2.296


In [10]:
print("Sample rows:")
display(verify.head(5))

Sample rows:


,timestamp,cmdb_id,case,container_cpu_usage_seconds_rate,container_cpu_system_seconds_rate,container_cpu_user_seconds_rate,container_memory_usage_bytes,container_memory_working_set_bytes,container_memory_rss,container_memory_cache,label,failure_type
0,1719252015,observe.cartservice-0,single_case1,2.242894,2.595977,2.197721,-1.252447,-1.25156,-1.301874,-0.612205,0,NaN
1,1719252030,observe.cartservice-0,single_case1,-0.471981,-0.462826,-0.448592,-1.252447,-1.25156,-1.301874,-0.612205,0,NaN
2,1719252045,observe.cartservice-0,single_case1,-0.471981,-0.462826,-0.448592,-1.252447,-1.25156,-1.301874,-0.612205,0,NaN
3,1719252060,observe.cartservice-0,single_case1,-0.471981,-0.462826,-0.448592,-1.252447,-1.25156,-1.301874,-0.612205,0,NaN
4,1719252075,observe.cartservice-0,single_case1,2.140858,1.127752,2.333429,-1.252447,-1.25156,-1.307516,-0.612205,0,NaN


## How to Split for Model Training

```python
df = pd.read_csv('data/processed/all_cases_labeled.csv')

# VAE training — normal behavior only from simplest case
train = df[(df['case'] == 'single_case1') & (df['label'] == 0)]

# Threshold tuning — use single_case2
val = df[df['case'] == 'single_case2']

# Drift evaluation — complex cases
test_drift = df[df['case'].str.startswith('complex')]
```